In [ ]:
"""
===============================================================================
Module: Environment Setup & Core Dependencies
Description: 
    Initializes the computational environment and establishes bindings for 
    the multi-agent simulation framework. This includes the physics engine 
    (HighwayEnv), neural embedding architectures, multi-provider LLM interfaces, 
    and statistical validation toolkits required for empirical execution.
Reproducibility:
    The requisite package ecosystem (documented below) must be installed 
    within the host environment to guarantee deterministic execution of the 
    ablation studies and baseline comparisons.
===============================================================================
"""
!pip install --upgrade pip
!pip install numpy highway-env gymnasium faiss-cpu sentence-transformers \
            groq openai anthropic pandas matplotlib seaborn \
            tqdm scikit-learn scipy joblib diskcache ipykernel

import os, json, time, random, re, warnings, hashlib, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Tuple, Optional, Any
from enum import Enum
from scipy import stats

import gymnasium as gym
import highway_env  # <--- ADD THIS LINE TO REGISTER THE ENVIRONMENTS

import faiss
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
print('✓ All packages imported and environments registered')

  Using cached pip-26.0.1-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-26.0.1-py3-none-any.whl (1.8 MB)


ERROR: To modify pip, please run the following command:
D:\Anaconda\envs\pymc_env\python.exe -m pip install --upgrade pip


  Using cached highway_env-1.10.2-py3-none-any.whl.metadata (16 kB)
  Using cached gymnasium-1.2.3-py3-none-any.whl.metadata (10 kB)
  Using cached sentence_transformers-5.2.3-py3-none-any.whl.metadata (16 kB)
  Using cached groq-1.0.0-py3-none-any.whl.metadata (16 kB)
  Using cached openai-2.21.0-py3-none-any.whl.metadata (29 kB)
  Using cached anthropic-0.83.0-py3-none-any.whl.metadata (28 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
  Using cached Farama_Notifications-0.0.4-py3-none-any.whl.metadata (558 bytes)
  Using cached transformers-5.2.0-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.4.1-py3-none-any.whl.metadata (13 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer_slim-0.24.0-py3-none-any.whl.met

d:\Anaconda\envs\pymc_env\lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
d:\Anaconda\envs\pymc_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ All packages imported and environments registered


In [ ]:
"""
===============================================================================
Module: Global Configuration & Experiment Parameters
Description: 
    Centralized configuration hub defining system hyperparameters, environment 
    variables, and runtime execution flags. This module directly governs the 
    global state of the multi-agent framework, including the dynamic 
    selection of the underlying LLM backend provider for comparative evaluations.
===============================================================================
"""

class LLMProvider(Enum):
    GROQ      = 'groq'
    OPENAI    = 'openai'
    ANTHROPIC = 'anthropic'
    MOCK      = 'mock'   # rule-based fallback — NOT for reported results

@dataclass
class APIConfig:
    # ── CHOOSE YOUR PROVIDER ──────────────────────────────────
    provider: LLMProvider = LLMProvider.GROQ   # change to OPENAI or ANTHROPIC

    # ── GROQ ─────────────────────────────────────────────────
    groq_api_key:  str = ''                    # or set env GROQ_API_KEY
    groq_model:    str = 'llama-3.1-8b-instant'

    # ── OPENAI ───────────────────────────────────────────────
    openai_api_key: str = ''                   # or set env OPENAI_API_KEY
    openai_model:   str = 'gpt-4o-mini'        # gpt-4o for best results

    # ── ANTHROPIC CLAUDE ─────────────────────────────────────
    anthropic_api_key: str = ''                # or set env ANTHROPIC_API_KEY
    anthropic_model:   str = 'claude-haiku-4-5-20251001'  # claude-opus-4-6 for best

    # ── SHARED ───────────────────────────────────────────────
    temperature:  float = 0.3   # lower = more deterministic = more reproducible
    max_tokens:   int   = 256

@dataclass
class FrameworkConfig:
    # ── ABLATION TOGGLES ────────────────────────────────────
    enable_master_agent:     bool = True
    enable_rag_verification: bool = True
    enable_reflection:       bool = True
    enable_intent_inference: bool = True

    # ── RAG — paper §III-E, Eq.7,9 ─────────────────────────
    k_candidates:  int   = 5
    k_final:       int   = 3
    tau_verify:    float = 0.5
    lambda_time:   float = 0.1
    embedding_dim: int   = 384

    # ── MASTER AGENT — Eq.5,6 ───────────────────────────────
    lambda_coop:     float = 0.3
    lambda_conflict: float = 0.5
    delta_t_safe:    float = 2.0

    # ── REFLECTION — Eq.14 ──────────────────────────────────
    gamma:         float = 0.99
    theta_high:    float = 0.7
    alignment_eps: float = 0.5

    # ── ENVIRONMENT ─────────────────────────────────────────
    num_agents:       int   = 2
    ramp_length:      float = 120.0
    init_speed_min:   float = 20.0
    init_speed_max:   float = 25.0
    policy_frequency: int   = 2
    max_steps:        int   = 60

    # ── EMBEDDING CACHE ─────────────────────────────────────
    embed_cache_dir: str = './embed_cache'

@dataclass
class ExperimentConfig:
    name:           str       = 'koma_rag_ieee'
    train_rounds:   int       = 5#40
    test_scenarios: int       = 3#20
    test_every:     int       = 3#20
    # !! Use 5+ seeds for IEEE submission
    seeds:          List[int] = field(default_factory=lambda: [42, 123, 456, 789, 999])
    save_results:   bool      = True
    results_dir:    str       = './results'

def setup_api(cfg: APIConfig) -> APIConfig:
    """Auto-load keys from environment or Colab secrets."""
    def _get(env_key, attr):
        val = os.environ.get(env_key, getattr(cfg, attr, ''))
        try:
            from google.colab import userdata
            val = val or userdata.get(env_key)
        except Exception:
            pass
        return val or ''

    cfg.groq_api_key      = _get('GROQ_API_KEY',      'groq_api_key')
    cfg.openai_api_key    = _get('OPENAI_API_KEY',    'openai_api_key')
    cfg.anthropic_api_key = _get('ANTHROPIC_API_KEY', 'anthropic_api_key')
    return cfg

api_config = setup_api(APIConfig())
fw_config  = FrameworkConfig()
exp_config = ExperimentConfig()

Path(exp_config.results_dir).mkdir(exist_ok=True)
Path(fw_config.embed_cache_dir).mkdir(exist_ok=True)

print(f'✓ Config ready  |  provider={api_config.provider.value}')
print(f'  Groq key  : {"SET" if api_config.groq_api_key else "MISSING"}')
print(f'  OpenAI key: {"SET" if api_config.openai_api_key else "MISSING"}')
print(f'  Claude key: {"SET" if api_config.anthropic_api_key else "MISSING"}')

✓ Config ready  |  provider=groq
  Groq key  : SET
  OpenAI key: SET
  Claude key: MISSING


In [ ]:
"""
===============================================================================
Module: Core Data Structures & State Representation
Description: 
    Defines the foundational data models, memory schemas, and state 
    representations utilized across the multi-agent framework. These classes 
    formally encode the episodic experiences, agent trajectories, and retrieval 
    interactions mathematically formulated in the accompanying manuscript.
===============================================================================
"""

class Action(Enum):
    IDLE              = 0
    ACCELERATE        = 1
    DECELERATE        = 2
    LANE_CHANGE_LEFT  = 3
    LANE_CHANGE_RIGHT = 4
    def __str__(self): return self.name

HW_ACTION_MAP = {
    Action.LANE_CHANGE_LEFT:  0,
    Action.IDLE:              1,
    Action.LANE_CHANGE_RIGHT: 2,
    Action.ACCELERATE:        3,
    Action.DECELERATE:        4,
}

@dataclass
class AgentState:
    """s_i(t) = [x_i, y_i, θ_i, v_i, a_i, l_i]^T  — paper Eq.3"""
    agent_id:     str
    x:            float
    y:            float
    theta:        float
    velocity:     float
    acceleration: float
    lane:         int
    priority:     float = 0.5
    is_ego:       bool  = True
    on_ramp:      bool  = False

    def to_vec(self) -> np.ndarray:
        return np.array([self.x, self.y, self.theta,
                         self.velocity, self.acceleration, float(self.lane)])

    def describe(self) -> str:
        p   = f'{self.priority:.2f}' if self.priority != float('inf') else '∞'
        loc = 'RAMP' if self.on_ramp else 'MAIN'
        return (f'[{self.agent_id}|{loc}] pos=({self.x:.1f},{self.y:.1f}) '
                f'lane={self.lane} vel={self.velocity:.1f}m/s '
                f'accel={self.acceleration:.2f}m/s² prio={p}')

@dataclass
class Experience:
    """e_k = (D_k, G_k, P_k, u_k, r_k, z_k)"""
    exp_id:           str
    description:      str
    goal:             str
    plan:             str
    action:           Action
    reward:           float
    safety_score:     float = 0.0
    efficiency_score: float = 0.0
    embedding:        Optional[np.ndarray] = None
    timestamp:        float = 0.0
    scenario_type:    str   = 'merge'
    verified:         bool  = False
    assigned_goal:    str   = ''
    # store raw V score for analysis
    v_score:          float = 0.0

    def to_dict(self) -> Dict:
        d = asdict(self)
        d['action']    = self.action.name
        d['embedding'] = None
        return d

@dataclass
class CoordinationMsg:
    target:      str
    goal:        str
    priority:    float
    constraints: List[str]
    timestamp:   float

@dataclass
class AgentReport:
    agent_id:     str
    proposed_goal:str
    state:        AgentState

print('✓ Data structures ready')

✓ Data structures ready


In [ ]:
"""
===============================================================================
Module: Cached Embedding Architecture
Description: 
    Implements a high-performance, dual-tier (in-memory and persistent disk) 
    caching mechanism for vector embeddings. This module optimizes 
    computational overhead by intercepting and eliminating redundant neural 
    network (e.g., SentenceTransformer) inference passes during memory retrieval.
Performance Characteristics:
    Substantially reduces embedding generation latency from baseline inference 
    times (~200ms) to sub-millisecond (<1ms) cache hits. This optimization is 
    critical for enabling scalable, multi-agent episodic rollouts without 
    continuous GPU/CPU bottlenecks.
===============================================================================
"""

class CachedEmbeddingModule:
    """
    Two-level cache:
      L1 — in-memory dict  (key = sha256 of text)
      L2 — disk pickle     (persists across notebook restarts)
    """
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2',
                 cache_dir: str = './embed_cache'):
        self._model_name = model_name
        self._cache_dir  = Path(cache_dir)
        self._cache_dir.mkdir(exist_ok=True)
        self._disk_path  = self._cache_dir / f'{model_name.replace("/","_")}.pkl'

        # L1: memory cache
        self._mem: Dict[str, np.ndarray] = {}

        # Load L2 disk cache if it exists
        if self._disk_path.exists():
            with open(self._disk_path, 'rb') as f:
                self._mem = pickle.load(f)
            print(f'  ✓ Loaded {len(self._mem)} cached embeddings from disk')

        # Lazy-load the model (only when actually needed)
        self._model = None
        self.dim    = 384   # all-MiniLM-L6-v2 fixed dim
        self._new_since_save = 0

    def _load_model(self):
        if self._model is None:
            print('  Loading SentenceTransformer (one-time)...')
            self._model = SentenceTransformer(self._model_name)
            self.dim    = self._model.get_sentence_embedding_dimension()
            print(f'  ✓ Model loaded, dim={self.dim}')

    @staticmethod
    def _key(text: str) -> str:
        return hashlib.sha256(text.encode('utf-8')).hexdigest()

    def embed(self, text: str) -> np.ndarray:
        k = self._key(text)
        if k in self._mem:
            return self._mem[k]
        self._load_model()
        vec = self._model.encode(text, normalize_embeddings=True).astype(np.float32)
        self._mem[k] = vec
        self._new_since_save += 1
        if self._new_since_save >= 50:   # flush every 50 new embeddings
            self._flush()
        return vec

    def embed_batch(self, texts: List[str]) -> np.ndarray:
        """Batch embed — only compute uncached texts."""
        keys    = [self._key(t) for t in texts]
        missing = [(i, t) for i, (k, t) in enumerate(zip(keys, texts))
                   if k not in self._mem]

        if missing:
            self._load_model()
            idxs, txts = zip(*missing)
            vecs = self._model.encode(list(txts),
                                      normalize_embeddings=True,
                                      batch_size=64).astype(np.float32)
            for i, k, v in zip(idxs, [keys[i] for i in idxs], vecs):
                self._mem[k] = v
                self._new_since_save += 1

        result = np.stack([self._mem[k] for k in keys])
        if self._new_since_save >= 50:
            self._flush()
        return result

    def _flush(self):
        with open(self._disk_path, 'wb') as f:
            pickle.dump(self._mem, f, protocol=4)
        self._new_since_save = 0

    def flush(self):
        """Public flush — call at end of experiment."""
        self._flush()
        print(f'  ✓ Flushed {len(self._mem)} embeddings to {self._disk_path}')

    @property
    def cache_size(self) -> int:
        return len(self._mem)

embedder = CachedEmbeddingModule(
    model_name='all-MiniLM-L6-v2',
    cache_dir=fw_config.embed_cache_dir
)
print(f'✓ CachedEmbeddingModule ready  |  cache_size={embedder.cache_size}')

  ✓ Loaded 361 cached embeddings from disk
✓ CachedEmbeddingModule ready  |  cache_size=361


In [ ]:
"""
===============================================================================
Module: Multi-Provider LLM Interface
Description: 
    Implements a dynamic, hot-swappable routing interface for Large Language 
    Models (LLMs). Supports scalable integration with Groq, OpenAI, and 
    Anthropic, alongside a Mock provider for deterministic ablation studies.
Configuration:
    Provider execution is governed by the global `api_config.provider` state.
===============================================================================
"""

try:
    from groq import Groq as GroqClient
    GROQ_OK = True
except ImportError:
    GROQ_OK = False

try:
    from openai import OpenAI as OAIClient
    OAI_OK = True
except ImportError:
    OAI_OK = False

try:
    import anthropic as AnthropicLib
    ANT_OK = True
except ImportError:
    ANT_OK = False


class LLMInterface:
    """
    Unified LLM interface.
    - Groq / OpenAI / Anthropic share the same .generate() API.
    - Mock is a transparent fallback for unit-testing only.
    """
    def __init__(self, cfg: APIConfig):
        self.cfg        = cfg
        self.call_count = 0
        self.total_tokens = 0
        self._client    = None
        self._active_provider = LLMProvider.MOCK

        if cfg.provider == LLMProvider.GROQ and GROQ_OK and cfg.groq_api_key:
            self._client = GroqClient(api_key=cfg.groq_api_key)
            self._active_provider = LLMProvider.GROQ
            print(f'  ✓ Provider: Groq  |  model={cfg.groq_model}')

        elif cfg.provider == LLMProvider.OPENAI and OAI_OK and cfg.openai_api_key:
            self._client = OAIClient(api_key=cfg.openai_api_key)
            self._active_provider = LLMProvider.OPENAI
            print(f'  ✓ Provider: OpenAI  |  model={cfg.openai_model}')

        elif cfg.provider == LLMProvider.ANTHROPIC and ANT_OK and cfg.anthropic_api_key:
            self._client = AnthropicLib.Anthropic(api_key=cfg.anthropic_api_key)
            self._active_provider = LLMProvider.ANTHROPIC
            print(f'  ✓ Provider: Anthropic  |  model={cfg.anthropic_model}')

        else:
            print('  ⚠  No valid API key found → MOCK LLM active.')
            print('     Results with MOCK must NOT be reported as LLM results.')

    @property
    def is_real_llm(self) -> bool:
        return self._active_provider != LLMProvider.MOCK

    # ─── public API ────────────────────────────────────────
    def generate(self, prompt: str, system: str = '',
                 json_mode: bool = False) -> str:
        self.call_count += 1
        if self._client is not None:
            try:
                return self._call_api(prompt, system, json_mode)
            except Exception as e:
                print(f'  ⚠ API error ({type(e).__name__}: {e}) — falling back to mock')
        return self._mock(prompt)

    def factual_check(self, memory_desc: str, current_ctx: str) -> float:
        """
        V_factual: ask LLM to rate how consistent the retrieved memory
        is with the current context. Returns [0,1].
        This is called for BOTH Base KoMA and KoMA-RAG so comparisons are fair.
        """
        prompt = (
            f'Memory: "{memory_desc[:200]}"\n'
            f'Current scenario: "{current_ctx[:200]}"\n\n'
            'Rate factual consistency 0.0-1.0. '
            'Consider: same road type, compatible speeds, compatible lane counts. '
            'Reply JSON only: {"score": 0.0}'
        )
        resp = self.generate(prompt, 'Rate consistency. JSON only.', json_mode=True)
        try:
            return float(json.loads(resp).get('score', 0.5))
        except Exception:
            # regex fallback
            m = re.search(r'[0-9]\.[0-9]+', resp)
            return float(m.group()) if m else 0.5

    # ─── provider implementations ──────────────────────────
    def _call_api(self, prompt: str, system: str, json_mode: bool) -> str:
        sys_msg = system or 'You are an expert autonomous driving agent.'

        if self._active_provider == LLMProvider.ANTHROPIC:
            # Anthropic uses a different API shape
            resp = self._client.messages.create(
                model   = self.cfg.anthropic_model,
                max_tokens = self.cfg.max_tokens,
                system  = sys_msg,
                messages = [{'role': 'user', 'content': prompt}]
            )
            self.total_tokens += (resp.usage.input_tokens +
                                   resp.usage.output_tokens)
            return resp.content[0].text

        # Groq and OpenAI share the OpenAI messages format
        msgs = [
            {'role': 'system', 'content': sys_msg},
            {'role': 'user',   'content': prompt}
        ]
        kw = dict(messages=msgs,
                  temperature=self.cfg.temperature,
                  max_tokens=self.cfg.max_tokens)

        if self._active_provider == LLMProvider.GROQ:
            kw['model'] = self.cfg.groq_model
            resp = self._client.chat.completions.create(**kw)
        else:  # OPENAI
            kw['model'] = self.cfg.openai_model
            if json_mode:
                kw['response_format'] = {'type': 'json_object'}
            resp = self._client.chat.completions.create(**kw)

        self.total_tokens += resp.usage.total_tokens if resp.usage else 0
        return resp.choices[0].message.content

    def _mock(self, prompt: str) -> str:
        """Deterministic rule-based mock. Only for unit-testing, never reported."""
        p      = prompt.lower()
        ramp_m = re.search(r'ramp.*?(\d+(?:\.\d+)?)m', p)
        vel_m  = re.search(r'vel.*?(\d+(?:\.\d+)?)m/s', p)
        gap_m  = re.search(r'(\d+(?:\.\d+)?)m (?:ahead|behind)', p)
        ramp   = float(ramp_m.group(1)) if ramp_m else 999.0
        vel    = float(vel_m.group(1))  if vel_m  else 22.0
        gap    = float(gap_m.group(1))  if gap_m  else 999.0

        if 'score' in p:   # factual check
            return json.dumps({'score': round(random.uniform(0.4, 0.8), 2)})
        if 'goal' in p:
            return json.dumps({'goal': 'Merge safely before ramp ends',
                               'reasoning': 'Ramp vehicle must merge'})
        if 'plan' in p:
            return json.dumps({'plan': ['Check gap in main lane',
                                        'Accelerate to match speed'],
                               'outcome': 'Safe merge'})
        # action selection
        if ramp < 30:
            a = 'LANE_CHANGE_LEFT'
        elif gap < 10:
            a = 'DECELERATE'
        elif ramp < 70 and gap > 35:
            a = 'LANE_CHANGE_LEFT'
        elif vel < 20:
            a = 'ACCELERATE'
        else:
            a = 'IDLE'
        return json.dumps({'action': a, 'confidence': 0.75})

    def stats(self) -> Dict:
        return {
            'provider':    self._active_provider.value,
            'is_real_llm': self.is_real_llm,
            'call_count':  self.call_count,
            'total_tokens': self.total_tokens,
        }

llm = LLMInterface(api_config)
print(f'✓ LLM ready  |  real={llm.is_real_llm}')

  ✓ Provider: Groq  |  model=llama-3.1-8b-instant
✓ LLM ready  |  real=True


In [ ]:
"""
===============================================================================
Module: High-Dimensional Vector Retrieval Engine (FAISS)
Description: 
    Implements an optimized, scalable vector database utilizing FAISS 
    (Facebook AI Similarity Search) for the storage and rapid retrieval of 
    agent memory embeddings. Facilitates efficient nearest-neighbor (k-NN) 
    similarity lookups to power the Retrieval-Augmented Generation (RAG) 
    mechanisms.
Integration:
    Interfaces directly with the Cached Embedding Architecture to index 
    episodic trajectories, enabling sub-linear search complexity during 
    real-time multi-agent simulation rollouts.
===============================================================================
"""

class FAISSStore:
    """Shared vector database — accessible by ALL agents (paper §III-D)."""
    def __init__(self, dim: int):
        self.index       = faiss.IndexFlatIP(dim)   # cosine on L2-normalised vecs
        self.experiences: List[Experience] = []

    def add(self, exp: Experience):
        if exp.embedding is not None:
            self.index.add(exp.embedding.reshape(1, -1))
            self.experiences.append(exp)

    def search(self, q: np.ndarray, k: int) -> List[Tuple[Experience, float]]:
        if self.index.ntotal == 0:
            return []
        k = min(k, self.index.ntotal)
        scores, ids = self.index.search(q.reshape(1, -1), k)
        return [(self.experiences[i], float(scores[0][j]))
                for j, i in enumerate(ids[0]) if i >= 0]

    def reset(self):
        """Clear store — used between ablation runs."""
        dim = self.index.d
        self.index = faiss.IndexFlatIP(dim)
        self.experiences.clear()

    @property
    def stats(self) -> Dict:
        verified = sum(1 for e in self.experiences if e.verified)
        return {'total': len(self.experiences), 'verified': verified}

store = FAISSStore(embedder.dim)
print(f'✓ FAISS store ready  |  dim={embedder.dim}')

✓ FAISS store ready  |  dim=384


In [ ]:
"""
===============================================================================
Module: Episodic Memory Retrieval & Verification Architecture
Reference: Methodology Section III-E (Eq. 7-9), Algorithm 1
Description: 
    Implements the core shared memory infrastructure for the multi-agent 
    framework. This encompasses both the vector-based retrieval engine 
    (RAGMemory) and the tripartite verification scoring system 
    (VerificationModule).
Verification Formulation:
    Calculates the composite verification score:
    $V(e_j, \Omega_i) = V_{semantic} \cdot V_{factual} \cdot V_{contextual}$
Experimental Control:
    To ensure rigorous comparative analysis, verification scores are computed 
    and logged symmetrically across both baseline (unfiltered nearest-neighbor) 
    and augmented (threshold-filtered, $\tau_{verify}$) configurations. This 
    standardizes the empirical tracking of retrieval quality and factual 
    consistency across all ablation studies.
===============================================================================
"""
@dataclass
class Experience:
    """e_k = (D_k, G_k, P_k, u_k, r_k, z_k) — paper Eq.3"""
    exp_id:           str
    description:      str
    goal:             str
    plan:             str
    action:           Action
    reward:           float
    agent_id:         str   = ''    # agent_id for reward lookup
    safety_score:     float = 0.0
    efficiency_score: float = 0.0
    embedding:        Optional[np.ndarray] = None
    timestamp:        float = 0.0
    scenario_type:    str   = 'merge'
    verified:         bool  = False
    assigned_goal:    str   = ''
    v_score:          float = 0.0

    def to_dict(self) -> Dict:
        d = asdict(self)
        d['action']    = self.action.name
        d['embedding'] = None
        return d


class VerificationModule:
    """
    V(e_j, Ω_i) = V_semantic · V_factual · V_contextual   Eq.8
    V_contextual = exp(−λ_t·|t_j−t|) · I[scenario_match]  Eq.9
    """
    def __init__(self, llm, cfg: FrameworkConfig):
        self.llm = llm
        self.cfg = cfg
        self._all_v_scores:   List[float] = []
        self._filtered_count: int         = 0
        self._passed_count:   int         = 0

    def compute_full(self, exp: Experience, ctx: str,
                     ctx_emb: np.ndarray, t: float) -> Tuple[float, Dict]:
        # V_semantic
        v_sem = float(np.clip(np.dot(exp.embedding, ctx_emb), 0.0, 1.0)) \
                if exp.embedding is not None else 0.0

        # V_factual via LLM
        v_fac = float(np.clip(
            self.llm.factual_check(exp.description, ctx), 0.0, 1.0
        ))

        # V_contextual  Eq.9
        decay = float(np.exp(-self.cfg.lambda_time * abs(exp.timestamp - t)))
        scene = 1.0 if exp.scenario_type == 'merge' else 0.5
        v_ctx = float(np.clip(decay * scene, 0.0, 1.0))

        composite = v_sem * v_fac * v_ctx
        self._all_v_scores.append(composite)
        return composite, {'v_sem': v_sem, 'v_fac': v_fac,
                           'v_ctx': v_ctx, 'composite': composite}

    def mean_v_score(self) -> float:
        return float(np.mean(self._all_v_scores)) if self._all_v_scores else 0.0

    def reset_stats(self):
        self._all_v_scores.clear()
        self._filtered_count = 0
        self._passed_count   = 0


class RAGMemory:
    """Shared memory — Algorithm 1 lines 14-19."""
    def __init__(self, store, emb, ver: VerificationModule,
                 cfg: FrameworkConfig):
        self.store = store
        self.emb   = emb
        self.ver   = ver
        self.cfg   = cfg
        self._retrieval_log: List[Dict] = []

    def retrieve(self, ctx: str, t: float,
                 apply_filter: bool = True) -> Tuple[List[Experience], float]:
        """
        returns (experiences, mean_v_score).
        mean_v_score is 0.0 when memory is empty (not None, not an error).
        V scores are measured in BOTH conditions for honest comparison.
        apply_filter=True  → KoMA-RAG  (filtered by τ_verify)
        apply_filter=False → Base KoMA (cosine-only)
        """
        z          = self.emb.embed(ctx)
        candidates = self.store.search(z, self.cfg.k_candidates)

        # if no candidates, return 0.0 explicitly (not accumulated)
        if not candidates:
            return [], 0.0

        scored = []
        for exp, sem_score in candidates:
            v, parts = self.ver.compute_full(exp, ctx, z, t)
            scored.append((exp, v, parts))

        # track mean_v per retrieval call, not globally
        mean_v = float(np.mean([s[1] for s in scored])) if scored else 0.0

        if apply_filter:
            passed = [(e, v) for e, v, _ in scored if v > self.cfg.tau_verify]
            passed.sort(key=lambda x: x[1], reverse=True)
            selected = [e for e, _ in passed[:self.cfg.k_final]]
            self.ver._passed_count   += len(selected)
            self.ver._filtered_count += len(scored) - len(selected)
        else:
            # Base KoMA: top-k by semantic score only (no V threshold)
            scored.sort(key=lambda x: x[1], reverse=True)
            selected = [e for e, _, _ in scored[:self.cfg.k_final]]

        self._retrieval_log.append({
            't': t, 'n_candidates': len(candidates),
            'n_selected': len(selected),
            'mean_v': mean_v, 'filtered': apply_filter
        })
        return selected, mean_v

    def store_experience(self, exp: Experience):
        exp.embedding = self.emb.embed(f'{exp.description} {exp.goal}')
        self.store.add(exp)

    def mean_retrieval_v(self) -> float:
        """
        average V score only over retrievals that had candidates.
        Empty-memory retrievals (returning 0.0) are excluded so early rounds
        don't drag the mean down artificially.
        """
        active = [r['mean_v'] for r in self._retrieval_log if r['n_candidates'] > 0]
        return float(np.mean(active)) if active else 0.0

    def reset_log(self):
        self._retrieval_log.clear()


# Rebuild
store      = FAISSStore(embedder.dim)
ver_module = VerificationModule(llm, fw_config)
memory     = RAGMemory(store, embedder, ver_module, fw_config)
print('✓ RAGMemory + VerificationModule ready (BUG 4 fixed)')

✓ RAGMemory + VerificationModule ready (BUG 4 fixed)


In [ ]:
"""
===============================================================================
Module: Multi-Agent Simulation Environment Wrapper
Reference: Experimental Setup & Scenario Definition
Description: 
    Provides a standardized, object-oriented interface to the underlying 
    kinematic simulation engine (HighwayEnv). This module manages the 
    instantiation of the multi-agent merging scenario, translating continuous 
    raw observation arrays into discrete, strongly-typed state representations 
    for the cognitive agents.
Empirical Tracking:
    Internally monitors and logs critical episodic performance metrics, 
    including collision events, successful topological merges, and 
    agent-specific reward trajectories required for final statistical evaluation.
===============================================================================
"""
LANE_WIDTH = 4.0

MERGE_CFG = {
    'observation': {
        'type': 'Kinematics',
        'vehicles_count': 8,
        'features': ['presence', 'x', 'y', 'vx', 'vy', 'heading'],
        'absolute': True,
        'normalize': False,
    },
    'action': {
        'type': 'MultiAgentAction',
        'action_config': {'type': 'DiscreteMetaAction'},
    },
    'controlled_vehicles': 2,
    'initial_vehicle_count': 5,
    'spawn_probability': 0.0,
    'duration': 30,
    'policy_frequency': 2,
    'simulation_frequency': 10,
    'collision_reward': -10,
    'high_speed_reward': 0.4,
    'lane_change_reward': -0.1,
    'reward_speed_range': [20, 30],
    'normalize_reward': False,
    'offroad_terminal': True,
}

class MergeEnvWrapper:
    def __init__(self, cfg: FrameworkConfig, seed: int = 42):
        self.cfg  = cfg
        self.seed = seed
        self._build()
        self.ego_agents:   Dict[str, AgentState]   = {}
        self.idm_vehicles: Dict[str, AgentState]   = {}
        self.timestep  = 0
        self.t         = 0.0
        self.dt        = 1.0 / cfg.policy_frequency
        self.done      = False
        self.collisions: Dict[str, bool]        = {}
        self.merges:     Dict[str, bool]        = {}
        self.rewards:    Dict[str, List[float]] = {}
        self._prev_vel:  Dict[str, Optional[float]] = {}
        self._ramp_end_x   = 250.0
        self._ramp_y_thresh = 6.0   # abs(y) > this → on ramp

        # BUG 2 FIX: track which agents started on ramp at episode reset
        self._started_on_ramp: Dict[str, bool] = {}

    def _build(self):
        try:
            self._env = gym.make('merge-v0', config=MERGE_CFG)
            self._env.unwrapped.configure(MERGE_CFG)
            self._env_name = 'merge-v0'
        except Exception as e:
            print(f'  ⚠ merge-v0 unavailable ({e}) → highway-v0')
            self._env = gym.make('highway-v0', config=MERGE_CFG)
            self._env.unwrapped.configure(MERGE_CFG)
            self._env_name = 'highway-v0'

    def _y_to_lane(self, y: float) -> int:
        return max(0, min(int(abs(y) / LANE_WIDTH), 3))

    def _is_on_ramp(self, y: float, x: float) -> bool:
        return abs(y) > self._ramp_y_thresh and x < self._ramp_end_x

    def _parse_row(self, agent_id: str, row: np.ndarray,
                   is_ego: bool = True) -> Tuple[Optional[AgentState], bool]:
        if float(row[0]) < 0.5:
            return None, False
        x, y   = float(row[1]), float(row[2])
        vx, vy = float(row[3]), float(row[4])
        hdg    = float(row[5]) if len(row) > 5 else 0.0
        vel    = float(np.sqrt(vx**2 + vy**2))
        prev   = self._prev_vel.get(agent_id)
        accel  = (vel - prev) / self.dt if prev is not None else 0.0
        self._prev_vel[agent_id] = vel
        lane    = self._y_to_lane(y)
        on_ramp = self._is_on_ramp(y, x)
        return AgentState(
            agent_id=agent_id, x=x, y=y, theta=hdg,
            velocity=vel, acceleration=accel, lane=lane,
            priority=float('inf') if not is_ego else 0.5,
            is_ego=is_ego, on_ramp=on_ramp
        ), True

    def reset(self, seed: Optional[int] = None) -> Dict:
        s = seed if seed is not None else self.seed
        obs, info = self._env.reset(seed=s)
        self.timestep = 0
        self.t        = 0.0
        self.done     = False
        self.collisions.clear()
        self.merges.clear()
        self.rewards.clear()
        self._prev_vel.clear()
        self._started_on_ramp.clear()   # BUG 2 FIX
        self._parse_obs(obs)

        # record which agents are on ramp at episode start
        for aid, st in self.ego_agents.items():
            self._started_on_ramp[aid] = st.on_ramp

        return self._scene_summary()

    def step(self, actions: Dict[str, Action]) -> Tuple[Dict, Dict[str, float], bool]:
        act_tuple = tuple(
            HW_ACTION_MAP.get(actions.get(aid, Action.IDLE), 1)
            for aid in sorted(self.ego_agents.keys())
        )
        obs, rew, term, trunc, info = self._env.step(act_tuple)
        self.timestep += 1
        self.t        += self.dt
        self.done      = bool(term or trunc)
        self._parse_obs(obs)

        rews = rew if isinstance(rew, (list, tuple)) else [rew, rew]
        per_agent = {}
        for i, aid in enumerate(sorted(self.ego_agents.keys())):
            r = float(rews[i]) if i < len(rews) else float(rews[0])
            self.rewards.setdefault(aid, []).append(r)
            per_agent[aid] = r

            # use ONLY info['crashed'] — never threshold on reward
            # highway-env sets info['crashed'] = True on actual collision
            crashed = info.get('crashed', False)
            # Also check per-agent crash info if available
            agents_info = info.get('agents_rewards', {})
            if not crashed and isinstance(agents_info, dict):
                crashed = any(
                    v <= -9.0   # collision_reward is -10, nothing else gets close
                    for v in agents_info.values()
                )
            if crashed:
                self.collisions[aid] = True

            # only count merge if agent STARTED on ramp AND is now on main
            s = self.ego_agents.get(aid)
            if s and self._started_on_ramp.get(aid, False):
                # Agent started on ramp — check if now on main lane
                now_on_main = abs(s.y) <= self._ramp_y_thresh and s.lane <= 1
                if now_on_main:
                    self.merges[aid] = True

        return self._scene_summary(), per_agent, self.done

    def _parse_obs(self, obs):
        self.ego_agents.clear()
        self.idm_vehicles.clear()
        if not isinstance(obs, (list, tuple, np.ndarray)):
            return
        obs_arr = np.array(obs)
        if obs_arr.ndim == 2:
            obs_arr = obs_arr[np.newaxis, ...]
        for i in range(min(len(obs_arr), 2)):
            agent_rows = obs_arr[i]
            if agent_rows.ndim >= 1 and len(agent_rows) > 0:
                row0 = agent_rows[0] if agent_rows.ndim > 1 else agent_rows
                aid  = f'ego_{i}'
                st, ok = self._parse_row(aid, row0, is_ego=True)
                if ok and st:
                    self.ego_agents[aid] = st
            if agent_rows.ndim > 1:
                for j, row in enumerate(agent_rows[1:5]):
                    vid = f'idm_{j}'
                    sv, ok = self._parse_row(vid, row, is_ego=False)
                    if ok and sv:
                        self.idm_vehicles[vid] = sv

    def _scene_summary(self) -> Dict:
        return {
            'ego': dict(self.ego_agents),
            'idm': dict(self.idm_vehicles),
            'ramp_dist': {
                aid: max(0.0, self._ramp_end_x - s.x)
                for aid, s in self.ego_agents.items() if s.on_ramp
            }
        }

    def get_desc(self) -> Dict:
        return self._scene_summary()

    def close(self):
        self._env.close()

print('✓ MergeEnvWrapper ready (BUG 1 + BUG 2 fixed)')

✓ MergeEnvWrapper ready (BUG 1 + BUG 2 fixed)


In [ ]:
"""
===============================================================================
Module: Safety & Efficiency Evaluation Metrics
Reference: Base KoMA Methodology (Section III-E)
Description: 
    Computes and aggregates quantitative performance metrics for the multi-agent 
    trajectories. This module operationalizes the system's safety constraints 
    (e.g., collision avoidance, critical headway) and efficiency objectives 
    (e.g., velocity maintenance, successful merge execution) into standardized 
    scoring functions.
Purpose:
    Provides the mathematical foundation for evaluating episodic success rates 
    and overall trajectory quality, enabling rigorous empirical comparisons 
    between the baseline architecture and the RAG-augmented configurations.
===============================================================================
"""

def compute_ttc(ego: AgentState, others: List[AgentState]) -> float:
    min_ttc = float('inf')
    for v in others:
        if abs(v.lane - ego.lane) > 0:
            continue
        rel_dist = v.x - ego.x
        if rel_dist <= 0:
            continue
        rel_vel = ego.velocity - v.velocity
        if rel_vel <= 0.5:
            continue
        ttc = rel_dist / rel_vel
        min_ttc = min(min_ttc, ttc)
    return min_ttc

def safety_score(ttc: float) -> float:
    """KoMA base paper formula."""
    if ttc > 3.0:  return 10.0
    if ttc < 1.5:  return 0.0
    return 20.0 * (ttc - 1.5) / 3.0

def efficiency_score(ego_vel: float, avg_vel: float) -> float:
    if avg_vel <= 0:
        return 10.0 if ego_vel >= 20 else 5.0
    return 10.0 if ego_vel >= avg_vel else 10.0 * ego_vel / avg_vel

def compute_scores(ego: AgentState,
                   others: List[AgentState]) -> Tuple[float, float]:
    ttc   = compute_ttc(ego, others)
    ss    = safety_score(ttc)
    avg_v = float(np.mean([v.velocity for v in others])) if others else ego.velocity
    es    = efficiency_score(ego.velocity, avg_v)
    return ss, es

print('✓ Scoring functions ready')

✓ Scoring functions ready


In [ ]:
"""
===============================================================================
Module: Cognitive Intent Inference Architecture
Description: 
    Implements the predictive intent modeling mechanisms for the ego agent. 
    This module analyzes the real-time kinematic states and historical 
    trajectories of surrounding entities to probabilistically infer their 
    underlying behavioral goals (e.g., merging, yielding, or maintaining velocity).
Integration:
    Serves as a critical prerequisite for proactive multi-agent coordination. 
    By anticipating dynamic environmental shifts, this module explicitly 
    conditions the Retrieval-Augmented Generation (RAG) context, enabling 
    more accurate and context-aware action selection.
===============================================================================
"""
class IntentModule:
    def __init__(self, llm: LLMInterface, cfg: FrameworkConfig):
        self.llm = llm
        self.cfg = cfg

    def infer(self, ego: AgentState,
              nearby: List[AgentState]) -> Dict[str, str]:
        if not self.cfg.enable_intent_inference:
            return {}
        return {v.agent_id: self._classify(ego, v) for v in nearby[:5]}

    @staticmethod
    def _classify(ego: AgentState, v: AgentState) -> str:
        if v.acceleration >  1.5: return 'aggressive_accel'
        if v.acceleration < -1.5: return 'hard_braking'
        rel = v.x - ego.x
        dv  = v.velocity - ego.velocity
        if rel > 0 and dv >  3: return 'faster_ahead'
        if rel < 0 and dv < -3: return 'slower_behind'
        if v.on_ramp:           return 'merging_vehicle'
        return 'steady_state'

    def context_summary(self, agent_id: str, ego: AgentState,
                        nearby: List[AgentState],
                        intents: Dict[str, str],
                        ramp_dist: float = -1.0) -> str:
        loc = 'RAMP' if ego.on_ramp else 'MAIN'
        rd  = f'ramp_remaining={ramp_dist:.0f}m ' if ramp_dist >= 0 else ''
        s   = (f'Ω_{agent_id}: loc={loc} vel={ego.velocity:.1f}m/s '
               f'lane={ego.lane} {rd}\n')
        for v in nearby[:4]:
            rel  = 'ahead' if v.x > ego.x else 'behind'
            dist = abs(v.x - ego.x)
            s += (
                f'  {v.agent_id}: {rel} {dist:.0f}m lane={v.lane} '
                f'vel={v.velocity:.1f} intent={intents.get(v.agent_id,"?")}\n'
            )
        return s.strip()

intent_module = IntentModule(llm, fw_config)
print('✓ IntentModule ready')

✓ IntentModule ready


In [ ]:
"""
===============================================================================
Module: Master Coordination Engine
Description: 
    Orchestrates the high-level cooperative policy and action selection 
    for the multi-agent system. By synthesizing inferred environmental intents, 
    kinematic state variables, and safety constraints, this module 
    mathematically formulates and resolves the decentralized coordination 
    problem to optimize joint trajectory outcomes.
Integration:
    Serves as the terminal decision-making locus, consuming outputs from 
    both the Cognitive Intent Inference and RAG Memory modules to emit 
    discrete, executable meta-actions into the simulation environment.
===============================================================================
"""

class MasterAgent:
    def __init__(self, llm: LLMInterface, cfg: FrameworkConfig):
        self.llm = llm
        self.cfg = cfg
        self.n_conflicts = 0
        self.n_resolved  = 0
        self.j_history: List[float] = []

    def _spatial_target(self, s: AgentState) -> Tuple[float, float, int]:
        h = self.cfg.delta_t_safe
        return s.x, s.x + s.velocity * h, s.lane

    def detect_conflicts(self, states: Dict[str, AgentState]) -> List[Tuple[str,str]]:
        """Eq.5: detect spatiotemporal trajectory overlaps."""
        aids = list(states.keys())
        conflicts = []
        for i, a1 in enumerate(aids):
            for a2 in aids[i+1:]:
                s1, s2 = states[a1], states[a2]
                x1s, x1e, l1 = self._spatial_target(s1)
                x2s, x2e, l2 = self._spatial_target(s2)
                spatial = (l1 == l2 and x1s < x2e and x2s < x1e)
                dv = abs(s1.velocity - s2.velocity)
                t_close = abs(s1.x - s2.x) / dv if dv > 0.1 else float('inf')
                if spatial and t_close < self.cfg.delta_t_safe:
                    conflicts.append((a1, a2))
                    self.n_conflicts += 1
        return conflicts

    def compute_priorities(self, states: Dict[str, AgentState],
                           idm_states: Dict[str, AgentState]) -> Dict[str, float]:
        """IDM vehicles get priority=inf (paper §III-D-2)."""
        p = {aid: float('inf') for aid in idm_states}
        for aid, s in states.items():
            base = min(1.0, 0.4 + 0.4 * (s.velocity / 30.0))
            if s.on_ramp:
                base = min(1.0, base + 0.3)
            p[aid] = base
        return p

    def compute_j(self, states: Dict[str, AgentState],
                  rewards: Dict[str, float],
                  has_conflict: bool) -> float:
        """Collective objective Eq.4."""
        r_sum = sum(rewards.values())
        avg_v = float(np.mean([s.velocity for s in states.values()])) if states else 0.0
        r_coop     = avg_v / 30.0
        r_conflict = 1.0 if has_conflict else 0.0
        J = r_sum + self.cfg.lambda_coop * r_coop - self.cfg.lambda_conflict * r_conflict
        self.j_history.append(J)
        return J

    def coordinate(self, ego_states: Dict[str, AgentState],
                   idm_states:  Dict[str, AgentState],
                   proposed_goals: Dict[str, str]) -> Dict[str, CoordinationMsg]:
        """Eq.6: assign directives resolving conflicts."""
        if not self.cfg.enable_master_agent:
            # Base KoMA: no master; pass-through with default goals
            return {
                aid: CoordinationMsg(
                    target=aid,
                    goal=proposed_goals.get(aid, 'Navigate merge safely'),
                    priority=0.5, constraints=[], timestamp=time.time())
                for aid in ego_states
            }

        priorities   = self.compute_priorities(ego_states, idm_states)
        conflicts    = self.detect_conflicts(ego_states)
        conflict_set = {aid for pair in conflicts for aid in pair}
        msgs = {}
        for aid, s in ego_states.items():
            goal        = proposed_goals.get(aid, 'Navigate merge safely')
            constraints = []
            if aid in conflict_set:
                other_ids = [b for a, b in conflicts if a == aid] + \
                            [a for a, b in conflicts if b == aid]
                for oid in other_ids:
                    op = priorities.get(oid, 0.5)
                    my_p = priorities.get(aid, 0.5)
                    if (op == float('inf')) or (op > my_p):
                        constraints.append(f'yield_to_{oid}')
                        if s.on_ramp:
                            goal = 'Slow down and wait for safe merge gap'
                self.n_resolved += 1
            msgs[aid] = CoordinationMsg(
                target=aid, goal=goal,
                priority=float(priorities.get(aid, 0.5)),
                constraints=constraints,
                timestamp=time.time()
            )
        return msgs

master_agent = MasterAgent(llm, fw_config)
print('✓ MasterAgent ready')

✓ MasterAgent ready


In [ ]:
"""
===============================================================================
Module: Hierarchical Planning Architecture
Description: 
    Implements the multi-tiered decision-making framework responsible for 
    translating high-level cognitive intents and RAG-augmented strategies 
    into executable, low-level control policies. This module bridges the 
    semantic reasoning space of the LLM with the discrete action space of 
    the kinematic simulation.
Execution:
    Governs the sequential mapping of coordinated meta-actions (e.g., yielding, 
    merging) into specific trajectory adjustments. Ensures that all generated 
    plans are mathematically validated against the physical and safety constraints 
    of the dynamic environment prior to execution.
===============================================================================
"""

class PlanningModule:
    """
    STAGE 1: g_i ← g_assigned      Eq.10
    STAGE 2: Plan with M_i(t)       Eq.11
    STAGE 3: Action s.t. C(u,msg)   Eq.12-13
    """
    def __init__(self, llm: LLMInterface, mem: RAGMemory, cfg: FrameworkConfig):
        self.llm = llm
        self.mem = mem
        self.cfg = cfg

    @staticmethod
    def _s(item) -> str:
        if isinstance(item, str):  return item
        if isinstance(item, dict): return str(next(iter(item.values()), ''))
        return str(item)

    def clarify_goal(self, ctx: str, assigned: str,
                     mems: List[Experience]) -> str:
        mem_txt = ''
        if mems:
            mem_txt = '\nPast goals: ' + '; '.join(
                f'"{m.goal[:40]}"' for m in mems[:2])
        prompt = (
            f'Merge scenario:\n{ctx[:300]}\n'
            f'Master assigned: {assigned}{mem_txt}\n\n'
            'Clarify goal. JSON only: {"goal": "...", "reasoning": "..."}'
        )
        r = self.llm.generate(prompt, 'Reply JSON only.', json_mode=True)
        try:   return json.loads(r).get('goal', assigned)
        except: return assigned

    def generate_plan(self, ctx: str, goal: str,
                      mems: List[Experience],
                      constraints: List[str]) -> List[str]:
        c_txt = ', '.join(constraints) or 'None'
        m_txt = ''
        if mems:
            m_txt = '\nPast plans:\n' + '\n'.join(
                f'  - {m.plan[:60]}' for m in mems[:2])
        prompt = (
            f'Goal: {goal}\nConstraints: {c_txt}{m_txt}\n\n'
            'Generate a 2-step merge plan. JSON only, items must be strings:\n'
            '{"plan": ["step one", "step two"], "outcome": "..."}'
        )
        r = self.llm.generate(prompt, 'Reply JSON only.', json_mode=True)
        try:
            raw = json.loads(r).get('plan', ['Maintain course'])
            return [self._s(i) for i in raw] or ['Maintain course']
        except:
            return ['Maintain course']

    def select_action(self, ctx: str, goal: str,
                      plan: List[str],
                      constraints: List[str]) -> Tuple[Action, float]:
        plan_str = '; '.join(self._s(p) for p in plan[:2])
        c_txt    = ', '.join(constraints) or 'None'
        prompt   = (
            f'Goal: {goal}\nPlan: {plan_str}\n'
            f'Master constraints: {c_txt}\n'
            f'Context (truncated):\n{ctx[:200]}\n\n'
            'Select ONE action. Exactly one of:\n'
            'IDLE, ACCELERATE, DECELERATE, LANE_CHANGE_LEFT, LANE_CHANGE_RIGHT\n'
            'JSON only: {"action": "NAME", "confidence": 0.0}'
        )
        r = self.llm.generate(
            prompt, 'Autonomous driving agent on highway merge.', json_mode=True)
        try:
            d   = json.loads(r)
            act = Action[d.get('action', 'IDLE').upper()]
            conf = float(d.get('confidence', 0.7))
            # Eq.13: consistency check against master constraints
            if 'yield' in c_txt and act == Action.LANE_CHANGE_LEFT:
                act = Action.DECELERATE
            return act, conf
        except:
            return Action.IDLE, 0.5

    def full_gpa(self, ctx: str, assigned_goal: str,
                 constraints: List[str], t: float,
                 apply_verification: bool = True
                 ) -> Tuple[str, List[str], Action, float, float]:
        """
        Goal-Plan-Action pipeline.  Returns:
          (goal, plan, action, confidence, mean_v_score)
        apply_verification controls whether retrieval filters by τ_verify.
        """
        mems, mean_v = self.mem.retrieve(ctx, t,
                                          apply_filter=apply_verification)
        goal   = self.clarify_goal(ctx, assigned_goal, mems)
        plan   = self.generate_plan(ctx, goal, mems, constraints)
        action, conf = self.select_action(ctx, goal, plan, constraints)
        return goal, plan, action, conf, mean_v

planner = PlanningModule(llm, memory, fw_config)
print('✓ PlanningModule ready')

✓ PlanningModule ready


In [ ]:
"""
===============================================================================
Module: Cognitive Reflection & Hindsight Evaluation
Description: 
    Executes the self-evaluative reflection mechanisms for the multi-agent 
    system. This module quantitatively and qualitatively assesses prior 
    meta-actions by comparing intended outcomes against the realized 
    kinematic states and safety metrics observed in the simulation environment.
Integration:
    Closes the cognitive feedback loop. By synthesizing short-term trajectory 
    feedback into generalized strategic insights, this module directly drives 
    the distillation and consolidation of high-value episodic experiences 
    into the long-term RAG retrieval corpus.
===============================================================================
"""

class ReflectionModule:
    """
    Store e_k iff r_k > θ_high AND Alignment(g_i, g_assigned) > ε  — Eq.14
    """
    def __init__(self, llm: LLMInterface, mem: RAGMemory,
                 emb: CachedEmbeddingModule, cfg: FrameworkConfig):
        self.llm = llm
        self.mem = mem
        self.emb = emb
        self.cfg = cfg
        self.stored_count    = 0
        self.reflection_count = 0

    def _alignment(self, goal: str, assigned: str) -> float:
        """Cosine similarity between goal and assigned_goal embeddings."""
        if not goal or not assigned:
            return 1.0
        g1 = self.emb.embed(goal)
        g2 = self.emb.embed(assigned)
        return float(np.clip(np.dot(g1, g2), 0.0, 1.0))

    def _reflect_and_correct(self, exp: Experience) -> Experience:
        """LLM rewrites the plan for a low-scoring experience."""
        self.reflection_count += 1
        prompt = (
            f'Episode description: {exp.description[:200]}\n'
            f'Goal: {exp.goal}\n'
            f'Original plan: {exp.plan}\n'
            f'Action taken: {exp.action}\n'
            f'Safety={exp.safety_score:.1f}/10  Efficiency={exp.efficiency_score:.1f}/10\n\n'
            'Suggest a corrected plan. JSON only: '
            '{"corrected_plan": "...", "reason": "..."}'
        )
        r = self.llm.generate(prompt, 'Driving reflection agent.', json_mode=True)
        try:
            d = json.loads(r)
            exp.plan = d.get('corrected_plan', exp.plan)
        except:
            pass
        return exp

    def process_episode(self, experiences: List[Experience]) -> int:
        """Reflect, then store qualifying experiences per Eq.14."""
        if not self.cfg.enable_reflection:
            # Store all high-reward experiences without reflection
            stored = 0
            for exp in experiences:
                if exp.reward > self.cfg.theta_high:
                    self.mem.store_experience(exp)
                    stored += 1
            self.stored_count += stored
            return stored

        stored = 0
        for exp in experiences:
            alignment = self._alignment(exp.goal, exp.assigned_goal)
            # Eq.14: r_k > θ_high AND alignment > ε
            if exp.reward > self.cfg.theta_high and alignment > self.cfg.alignment_eps:
                exp.verified = True
                self.mem.store_experience(exp)
                stored += 1
            elif (exp.safety_score < 3.0 or
                  exp.reward < 0.3 * self.cfg.theta_high):
                # Reflect on poor experiences and store corrected version
                corrected = self._reflect_and_correct(exp)
                if corrected.reward > 0.5 * self.cfg.theta_high:
                    corrected.verified = True
                    self.mem.store_experience(corrected)
                    stored += 1
        self.stored_count += stored
        return stored

reflection = ReflectionModule(llm, memory, embedder, fw_config)
print('✓ ReflectionModule ready')

✓ ReflectionModule ready


In [ ]:
"""
===============================================================================
Module: Core Framework Orchestration & Episodic Execution Loop
Reference: System Architecture & Experimental Setup
Description: 
    Serves as the central orchestration engine for the multi-agent framework. 
    This module structurally integrates the hierarchical cognitive components 
    (Intent Inference, Master Coordination, Planning, and Reflection) with the 
    kinematic simulation wrapper to execute closed-loop episodic rollouts.
Execution:
    Manages the overarching simulation protocols, periodic benchmark testing, 
    and the continuous aggregation of multi-dimensional performance metrics 
    (safety, efficiency, collision rates, and composite verification scores) 
    required for rigorous statistical validation.
===============================================================================
"""
@dataclass
class RoundResult:
    round_id:         int
    success:          bool
    collision:        bool
    merged:           bool
    total_reward:     float
    avg_safety:       float
    avg_efficiency:   float
    j_collective:     float
    steps:            int
    mean_v_score:     float
    actions:          List[str] = field(default_factory=list)
    decision_trace:   List[Dict] = field(default_factory=list)


class KoMARAG:
    def __init__(self, fw_cfg: FrameworkConfig,
                 api_cfg:    APIConfig,
                 shared_mem: RAGMemory):
        self.fw_cfg = fw_cfg
        self.llm    = LLMInterface(api_cfg)
        self.mem    = shared_mem
        self.intent = IntentModule(self.llm, fw_cfg)
        self.master = MasterAgent(self.llm, fw_cfg)
        self.plan   = PlanningModule(self.llm, self.mem, fw_cfg)
        self.refl   = ReflectionModule(self.llm, self.mem, embedder, fw_cfg)
        self.rounds: List[RoundResult] = []

    def _run_episode(self, env: MergeEnvWrapper,
                     seed: int) -> RoundResult:
        scene     = env.reset(seed=seed)
        all_safety: List[float]   = []
        all_effic:  List[float]   = []
        all_v:      List[float]   = []
        ep_exps:    List[Experience] = []
        trace:      List[Dict]    = []
        total_r   = 0.0
        apply_vfy = self.fw_cfg.enable_rag_verification
        ego_states = {}   # kept in outer scope for compute_j below

        for step in range(self.fw_cfg.max_steps):
            if env.done:
                break

            ego_states = scene['ego']
            idm_states = scene['idm']
            ramp_dists = scene.get('ramp_dist', {})

            if not ego_states:
                break

            # Master coordination
            proposals = {}
            for aid, s in ego_states.items():
                rd = ramp_dists.get(aid, -1.0)
                if s.on_ramp and rd >= 0:
                    proposals[aid] = f'Merge before ramp ends ({rd:.0f}m remaining)'
                elif s.on_ramp:
                    proposals[aid] = 'Merge onto main highway safely'
                else:
                    proposals[aid] = 'Maintain safe efficient cruising speed'

            coord_msgs = self.master.coordinate(
                ego_states, idm_states, proposals)

            # Per-agent GPA decisions
            actions: Dict[str, Action] = {}
            step_exps: List[Experience] = []   # experiences created THIS step

            for aid, s in ego_states.items():
                nearby  = list(idm_states.values())
                intents = self.intent.infer(s, nearby)
                rd      = ramp_dists.get(aid, -1.0)
                ctx     = self.intent.context_summary(aid, s, nearby, intents, rd)
                msg     = coord_msgs.get(aid)
                constraints = msg.constraints if msg else []
                assigned_g  = msg.goal if msg else proposals[aid]

                goal, plan, action, conf, mean_v = self.plan.full_gpa(
                    ctx, assigned_g, constraints, env.t,
                    apply_verification=apply_vfy
                )
                actions[aid] = action
                all_v.append(mean_v)

                ss, es = compute_scores(s, nearby + list(ego_states.values()))
                all_safety.append(ss)
                all_effic.append(es)

                exp = Experience(
                    exp_id=f'ep{len(self.rounds)}_s{step}_{aid}',
                    description=ctx[:200],
                    goal=goal,
                    plan='; '.join(plan[:2]),
                    action=action,
                    reward=0.0,          # filled after env.step()
                    agent_id=aid,        # store agent_id here
                    safety_score=ss,
                    efficiency_score=es,
                    timestamp=env.t,
                    scenario_type='merge',
                    assigned_goal=assigned_g,
                )
                step_exps.append(exp)
                ep_exps.append(exp)

                trace.append({
                    'step': step, 'agent': aid,
                    'context': ctx[:100],
                    'assigned_goal': assigned_g,
                    'clarified_goal': goal,
                    'plan': plan[:2],
                    'action': str(action),
                    'confidence': conf,
                    'mean_v_score': round(mean_v, 3),
                    'safety': round(ss, 2),
                    'efficiency': round(es, 2),
                })

            # Environment step
            scene, per_agent_r, done = env.step(actions)
            r_step = sum(per_agent_r.values())
            total_r += r_step

            # use exp.agent_id (e.g. 'ego_0') to look up reward
            # per_agent_r keys ARE agent IDs: {'ego_0': 0.3, 'ego_1': 0.2}
            for exp in step_exps:
                exp.reward = per_agent_r.get(
                    exp.agent_id,
                    r_step / max(len(per_agent_r), 1)   # safe fallback
                )

        # Reflection + memory
        self.refl.process_episode(ep_exps)

        j = self.master.compute_j(
            ego_states,
            {aid: total_r for aid in ego_states},
            has_conflict=(self.master.n_conflicts > 0)
        )

        collision = any(env.collisions.values())
        merged    = any(env.merges.values())
        success   = merged and not collision

        # BUG 4 FIX: use per-episode V score, not global accumulator
        ep_v_scores = [v for v in all_v if v > 0.0]  # exclude empty-memory 0.0s
        mean_v_ep   = float(np.mean(ep_v_scores)) if ep_v_scores else 0.0

        return RoundResult(
            round_id=len(self.rounds),
            success=success,
            collision=collision,
            merged=merged,
            total_reward=total_r,
            avg_safety=float(np.mean(all_safety)) if all_safety else 0.0,
            avg_efficiency=float(np.mean(all_effic)) if all_effic else 0.0,
            j_collective=j,
            steps=step + 1,
            mean_v_score=mean_v_ep,
            actions=[str(a) for a in actions.values()],
            decision_trace=trace
        )

    def train(self, env: MergeEnvWrapper,
              n_rounds: int = 10,
              test_every: int = 5,
              test_scenarios: int = 5,
              base_seed: int = 42) -> Dict:
        checkpoints = {}
        test_seeds  = list(range(base_seed + 1000, base_seed + 1000 + test_scenarios))

        for rnd in tqdm(range(n_rounds), desc='Training'):
            result = self._run_episode(env, seed=base_seed + rnd)
            self.rounds.append(result)

            if (rnd + 1) % test_every == 0:
                sr = self._test(env, test_seeds)
                checkpoints[rnd + 1] = sr
                print(f'  Checkpoint @{rnd+1}: success_rate={sr:.1%}  '
                      f'collision_rate={np.mean([r.collision for r in self.rounds]):.1%}  '
                      f'mem={self.mem.store.stats}  '
                      f'mean_v={np.mean([r.mean_v_score for r in self.rounds if r.mean_v_score>0]):.3f}'
                      if any(r.mean_v_score > 0 for r in self.rounds) else
                      f'  Checkpoint @{rnd+1}: success_rate={sr:.1%}  '
                      f'collision_rate={np.mean([r.collision for r in self.rounds]):.1%}  '
                      f'mem={self.mem.store.stats}  mean_v=N/A(memory empty)')

        return {'checkpoints': checkpoints, 'rounds': self.rounds}

    def _test(self, env: MergeEnvWrapper, test_seeds: List[int]) -> float:
        successes = sum(
            1 for s in test_seeds if self._run_episode(env, seed=s).success
        )
        return successes / len(test_seeds)

    def summary(self) -> Dict:
        if not self.rounds:
            return {}
        v_scores_nonzero = [r.mean_v_score for r in self.rounds if r.mean_v_score > 0]
        return {
            'n_rounds':        len(self.rounds),
            'success_rate':    float(np.mean([r.success    for r in self.rounds])),
            'collision_rate':  float(np.mean([r.collision  for r in self.rounds])),
            'mean_reward':     float(np.mean([r.total_reward for r in self.rounds])),
            'mean_safety':     float(np.mean([r.avg_safety   for r in self.rounds])),
            'mean_efficiency': float(np.mean([r.avg_efficiency for r in self.rounds])),
            # only average over rounds where retrieval happened
            'mean_v_score':    float(np.mean(v_scores_nonzero)) if v_scores_nonzero else 0.0,
            'llm_calls':       self.llm.call_count,
            'is_real_llm':     self.llm.is_real_llm,
        }

print('✓ KoMARAG ready')

✓ KoMARAG ready (BUG 3 + BUG 4 fixed)


In [ ]:
"""
===============================================================================
Module: Ablation Study & Comparative Evaluation Framework
Reference: Experimental Design & Results Analysis
Description: 
    Orchestrates the systematic ablation experiments to isolate and quantify 
    the performance contributions of the Retrieval-Augmented Generation (RAG) 
    and cognitive verification modules.
Experimental Controls:
    1. Cold-Start Initialization: All architectural variants strictly initialize 
       with an empty episodic memory store, precluding historical bias and 
       ensuring a rigorous zero-shot evaluation baseline.
    2. Symmetric Metric Tracking: The composite verification function 
       is continuously evaluated across all configurations. While the baseline 
       architecture bypasses threshold filtering, symmetric tracking guarantees 
       a statistically rigorous comparison of latent retrieval quality between 
       the standard and RAG-augmented configurations.
===============================================================================
"""

ABLATION_CONFIGS = {
    'Base_KoMA': dict(
        enable_master_agent=False,
        enable_rag_verification=False,
        enable_reflection=True,
        enable_intent_inference=True
    ),
    'KoMA_Master': dict(
        enable_master_agent=True,
        enable_rag_verification=False,
        enable_reflection=True,
        enable_intent_inference=True
    ),
    'KoMA_Verification': dict(
        enable_master_agent=False,
        enable_rag_verification=True,
        enable_reflection=True,
        enable_intent_inference=True
    ),
    'KoMA_RAG_Full': dict(
        enable_master_agent=True,
        enable_rag_verification=True,
        enable_reflection=True,
        enable_intent_inference=True
    ),
}

CFG_LABELS = {
    'Base_KoMA':       'Base KoMA',
    'KoMA_Master':     'KoMA + Master',
    'KoMA_Verification':'KoMA + Verification',
    'KoMA_RAG_Full':   'KoMA-RAG (Full)',
}
COLORS = {
    'Base_KoMA':        '#7f8c8d',
    'KoMA_Master':      '#27ae60',
    'KoMA_Verification':'#2980b9',
    'KoMA_RAG_Full':    '#e74c3c',
}

def run_ablation_config(cfg_name: str, ablation_flags: Dict,
                         seed: int = 42) -> Dict:
    """Run one ablation config from empty memory."""
    # Fresh store + memory for each config
    fresh_store = FAISSStore(embedder.dim)
    fresh_ver   = VerificationModule(llm, fw_config)
    fresh_mem   = RAGMemory(fresh_store, embedder, fresh_ver, fw_config)

    # Apply ablation flags
    ab_cfg = FrameworkConfig(**{
        **asdict(fw_config),
        **ablation_flags
    })

    env = MergeEnvWrapper(ab_cfg, seed=seed)
    fw  = KoMARAG(ab_cfg, api_config, fresh_mem)

    # Run 40 training rounds, checkpoint @20 and @40
    out = fw.train(env, n_rounds=exp_config.train_rounds,
                   test_every=exp_config.test_every,
                   test_scenarios=exp_config.test_scenarios,
                   base_seed=seed)
    env.close()
    embedder.flush()

    rounds = fw.rounds
    v_scores_base = [r.mean_v_score for r in rounds]

    return {
        'cfg_name':         cfg_name,
        'checkpoints':      out['checkpoints'],
        'success_0ep':      out['checkpoints'].get(0, 0.0),
        'success_20ep':     out['checkpoints'].get(20, 0.0),
        'success_40ep':     out['checkpoints'].get(40, 0.0),
        'collision_rate':   float(np.mean([r.collision for r in rounds])),
        'avg_safety':       float(np.mean([r.avg_safety for r in rounds])),
        'avg_efficiency':   float(np.mean([r.avg_efficiency for r in rounds])),
        'factual_consistency': float(np.mean(v_scores_base)),
        'mean_v_score':     float(np.mean(v_scores_base)),
        'rounds':           rounds,
        'llm_is_real':      fw.llm.is_real_llm,
        'llm_calls':        fw.llm.call_count,
        'decision_trace':   rounds[-1].decision_trace if rounds else [],
    }

print('✓ Ablation runner ready')
print(f'  Configs to run: {list(ABLATION_CONFIGS.keys())}')
print(f'  NOTE: All configs start from EMPTY shared memory — fair comparison')

✓ Ablation runner ready
  Configs to run: ['Base_KoMA', 'KoMA_Master', 'KoMA_Verification', 'KoMA_RAG_Full']
  NOTE: All configs start from EMPTY shared memory — fair comparison


In [ ]:
"""
===============================================================================
Module: Empirical Execution & Ablation Rollout
Reference: Experimental Setup & Results
Description: 
    Initiates the primary empirical execution phase of the study. This module 
    iterates through the predefined ablation configurations (e.g., Baseline 
    multi-agent vs. RAG-Augmented), systematically executing the episodic 
    rollouts across controlled, multi-seed validation environments.
Purpose:
    Serves as the main experimental driver, aggregating quantitative 
    performance trajectories, safety metrics, and qualitative decision traces 
    required for subsequent statistical validation and comparative analysis.
===============================================================================
"""
random.seed(42)
np.random.seed(42)

ab_results: Dict[str, Dict] = {}

for cfg_name, flags in ABLATION_CONFIGS.items():
    print(f'\n{"="*55}')
    print(f'  Running: {CFG_LABELS[cfg_name]}')
    print(f'  Flags: {flags}')
    print(f'{"="*55}')
    res = run_ablation_config(cfg_name, flags, seed=42)
    ab_results[cfg_name] = res
    print(f'  Done. success@40={res["success_40ep"]:.1%}  '
          f'factual_consistency={res["factual_consistency"]:.3f}  '
          f'collision={res["collision_rate"]:.1%}')

print('\n✓ Ablation complete')


  Running: Base KoMA
  Flags: {'enable_master_agent': False, 'enable_rag_verification': False, 'enable_reflection': True, 'enable_intent_inference': True}
  ✓ Provider: Groq  |  model=llama-3.1-8b-instant


Training:  20%|██        | 1/5 [00:20<01:20, 20.10s/it]

  Loading SentenceTransformer (one-time)...


Loading weights: 100%|██████████| 103/103 [00:02<00:00, 48.37it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ✓ Model loaded, dim=384


...

  Checkpoint @3: success_rate=0.0%  collision_rate=100.0%  mem={'total': 108, 'verified': 108}  mean_v=N/A(memory empty)


Training: 100%|██████████| 5/5 [2:20:38<00:00, 1687.64s/it]


  ✓ Flushed 452 embeddings to embed_cache\all-MiniLM-L6-v2.pkl
  Done. success@40=0.0%  factual_consistency=0.000  collision=100.0%

  Running: KoMA + Master
  Flags: {'enable_master_agent': True, 'enable_rag_verification': False, 'enable_reflection': True, 'enable_intent_inference': True}
  ✓ Provider: Groq  |  model=llama-3.1-8b-instant


Training:  40%|████      | 2/5 [1:29:13<2:28:37, 2972.49s/it]

  ⚠ API error (APIConnectionError: Connection error.) — falling back to mock
  ⚠ API error (APIConnectionError: Connection error.) — falling back to mock
  ⚠ API error (APIConnectionError: Connection error.) — falling back to mock
  ⚠ API error (APIConnectionError: Connection error.) — falling back to mock
  ⚠ API error (APIConnectionError: Connection error.) — falling back to mock


In [ ]:
"""
===============================================================================
Module: Statistical Validation & Hypothesis Testing
Reference: Experimental Results & Analysis
Description: 
    Executes rigorous statistical validation protocols to evaluate the 
    empirical performance differentials between the baseline multi-agent 
    architecture and the RAG-augmented framework. This module automates 
    multi-seed (n=5) validation rollouts to compute mean trajectory metrics 
    and robust confidence intervals.
Significance Testing:
    Conducts paired Student's t-tests across critical performance indicators 
    (e.g., collision rates, topological merge success, and composite V-scores) 
    to establish the formal statistical significance of the proposed 
    architectural enhancements, ensuring compliance with rigorous academic 
    publication standards.
===============================================================================
"""

STAT_SEEDS   = exp_config.seeds   
STAT_CONFIGS = ['Base_KoMA', 'KoMA_RAG_Full']

stat_results: Dict[str, List[float]] = {c: [] for c in STAT_CONFIGS}
stat_v:       Dict[str, List[float]] = {c: [] for c in STAT_CONFIGS}

print(f'Running {len(STAT_SEEDS)} seeds × {len(STAT_CONFIGS)} configs...')
for seed in STAT_SEEDS:
    for cfg_name in STAT_CONFIGS:
        flags = ABLATION_CONFIGS[cfg_name]
        r     = run_ablation_config(cfg_name, flags, seed=seed)
        stat_results[cfg_name].append(r['success_40ep'])
        stat_v[cfg_name].append(r['factual_consistency'])
        print(f'  seed={seed}  cfg={cfg_name}  '
              f'success={r["success_40ep"]:.1%}  v={r["factual_consistency"]:.3f}')

# t-tests
base_sr  = stat_results['Base_KoMA']
full_sr  = stat_results['KoMA_RAG_Full']
base_v   = stat_v['Base_KoMA']
full_v   = stat_v['KoMA_RAG_Full']

t_sr, p_sr = stats.ttest_ind(full_sr, base_sr)
t_v,  p_v  = stats.ttest_ind(full_v,  base_v)

print('\n' + '='*55)
print('  STATISTICAL RESULTS')
print('='*55)
for cfg in STAT_CONFIGS:
    sr_arr = np.array(stat_results[cfg])
    v_arr  = np.array(stat_v[cfg])
    print(f'  {CFG_LABELS[cfg]}:')
    print(f'    Success @40: {sr_arr.mean():.3f} ± {sr_arr.std():.3f}'
          f'  95%CI=[{sr_arr.mean()-1.96*sr_arr.std()/np.sqrt(len(sr_arr)):.3f},'
          f'{sr_arr.mean()+1.96*sr_arr.std()/np.sqrt(len(sr_arr)):.3f}]')
    print(f'    Mean V score: {v_arr.mean():.3f} ± {v_arr.std():.3f}')

print(f'\n  t-test (success rate): t={t_sr:.3f}, p={p_sr:.4f}  '
      f'{"SIGNIFICANT" if p_sr < 0.05 else "NOT significant"}')
print(f'  t-test (V score):      t={t_v:.3f}, p={p_v:.4f}  '
      f'{"SIGNIFICANT" if p_v < 0.05 else "NOT significant"}')

if not llm.is_real_llm:
    print('\n  ⚠ WARNING: Results above use MOCK LLM.')
    print('    Replace with a real LLM before citing in paper.')

In [ ]:
"""
===============================================================================
Module: Qualitative Case Study & Episodic Trajectory Analysis
Reference: Experimental Results & Discussion
Description: 
    Extracts and visualizes high-resolution, step-by-step decision traces 
    from selected simulation episodes. This module facilitates micro-level 
    qualitative analysis of agent reasoning, intent inference, and sequential 
    action selection.
Purpose:
    Provides interpretable, transparent case studies of the multi-agent 
    coordination dynamics. These granular traces directly complement the 
    aggregate statistical findings by offering concrete, verifiable examples 
    of RAG-augmented behavioral improvements and strategic planning.
===============================================================================
"""

print('='*65)
print('  QUALITATIVE CASE STUDY — Decision Traces')
print('='*65)

for cfg_name in ['Base_KoMA', 'KoMA_RAG_Full']:
    rounds = ab_results[cfg_name].get('rounds', [])
    if not rounds:
        continue

    # Pick last training round for both
    last = rounds[-1]
    print(f'\n--- {CFG_LABELS[cfg_name]}  '
          f'(Round {last.round_id}, success={last.success}) ---')
    print(f'  Total reward: {last.total_reward:.3f}  '
          f'Safety: {last.avg_safety:.2f}/10  '
          f'Efficiency: {last.avg_efficiency:.2f}/10  '
          f'Mean V: {last.mean_v_score:.3f}')

    trace = last.decision_trace
    # Show first 3 timesteps
    for entry in trace[:3]:
        print(f'\n  t={entry["step"]}  agent={entry["agent"]}')
        print(f'    Context:        {entry["context"][:80]}...')
        print(f'    Assigned goal:  {entry["assigned_goal"][:60]}')
        print(f'    Clarified goal: {entry["clarified_goal"][:60]}')
        plan_str = ' → '.join(entry['plan'][:2])
        print(f'    Plan:           {plan_str[:80]}')
        print(f'    Action:         {entry["action"]}  '
              f'(confidence={entry["confidence"]:.2f})')
        print(f'    V_score:        {entry["mean_v_score"]}  '
              f'Safety={entry["safety"]}  Efficiency={entry["efficiency"]}')

  QUALITATIVE CASE STUDY — Decision Traces


NameError: name 'ab_results' is not defined

In [ ]:
"""
===============================================================================
Module: High-Fidelity Data Visualization & Figure Generation
Reference: Experimental Results & Performance Metrics
Description: 
    Synthesizes the aggregated multi-seed experimental data into publication-
    ready graphical representations. This module plots key comparative 
    performance indicators, including episodic reward trajectories, safety 
    and efficiency curves, and ablation success rates.
===============================================================================
"""

plt.rcParams.update({
    'font.family': 'serif', 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.dpi': 150
})

def smooth(arr, w=5):
    if len(arr) < w:
        return arr
    return np.convolve(arr, np.ones(w)/w, mode='valid')

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for cfg_name, res in ab_results.items():
    rounds = res.get('rounds', [])
    if not rounds:
        continue
    label  = CFG_LABELS[cfg_name]
    color  = COLORS[cfg_name]
    lw     = 2.5 if 'Full' in cfg_name else 1.5

    rewards    = smooth([r.total_reward   for r in rounds])
    collisions = smooth([float(r.collision) for r in rounds])
    v_scores   = smooth([r.mean_v_score   for r in rounds])
    xs = range(len(rewards))

    axes[0].plot(xs, rewards,    color=color, lw=lw, label=label)
    axes[1].plot(xs, collisions, color=color, lw=lw, label=label)
    axes[2].plot(xs, v_scores,   color=color, lw=lw, label=label)

axes[0].set_title('Cumulative Reward'); axes[0].set_xlabel('Episode')
axes[1].set_title('Collision Rate');   axes[1].set_xlabel('Episode')
axes[2].set_title('Mean V-Score (Factual Consistency)'); axes[2].set_xlabel('Episode')

axes[2].set_ylabel('Mean composite V(e_j, Ω)')
axes[1].set_ylim(-0.05, 1.05)

for ax in axes:
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, linestyle='--')

plt.suptitle('KoMA-RAG vs Ablations — Performance Trends', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{exp_config.results_dir}/fig_performance.pdf', bbox_inches='tight')
plt.savefig(f'{exp_config.results_dir}/fig_performance.png', bbox_inches='tight')
plt.show()
print('✓ Figure saved')

In [ ]:
"""
===============================================================================
Module: Retrieval Quality & V-Score Distribution Analysis
Reference: Factual Consistency Evaluation
Description: 
    Computes and visualizes the comparative statistical distribution of 
    composite verification (V) scores across all experimental configurations. 
    This module provides the rigorous empirical foundation required to 
    substantiate claims of enhanced factual consistency within the system's 
    memory retrieval processes.
Empirical Objective:
    Demonstrates quantitatively that the threshold-filtered retrieval 
    mechanism (KoMA-RAG) systematically isolates higher-quality, more 
    contextually relevant episodic memories compared to the unconstrained 
    baseline architecture.
===============================================================================
"""

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

compare_cfgs = ['Base_KoMA', 'KoMA_Verification', 'KoMA_RAG_Full']

# Left: V-score histogram
for cfg_name in compare_cfgs:
    rounds = ab_results.get(cfg_name, {}).get('rounds', [])
    if not rounds: continue
    v_vals = [r.mean_v_score for r in rounds]
    axes[0].hist(v_vals, bins=15, alpha=0.55,
                 label=CFG_LABELS[cfg_name],
                 color=COLORS[cfg_name], density=True)

axes[0].set_xlabel('Mean Composite V-Score per Episode')
axes[0].set_ylabel('Density')
axes[0].set_title('Distribution of Retrieval Quality\n(higher = more factually consistent memories)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3, linestyle='--')

# Right: Success rate at checkpoints (bar chart — Table III)
checkpoints = [20, 40]
x = np.arange(len(checkpoints))
width = 0.2
for i, (cfg_name, res) in enumerate(ab_results.items()):
    vals = [res['checkpoints'].get(ck, 0.0) for ck in checkpoints]
    axes[1].bar(x + i*width, vals, width,
                label=CFG_LABELS[cfg_name], color=COLORS[cfg_name], alpha=0.85)

axes[1].set_xticks(x + width * 1.5)
axes[1].set_xticklabels([f'@{ck} rounds' for ck in checkpoints])
axes[1].set_ylabel('Success Rate')
axes[1].set_title('Success Rate at Training Checkpoints')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3, linestyle='--', axis='y')
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig(f'{exp_config.results_dir}/fig_v_score_comparison.pdf', bbox_inches='tight')
plt.savefig(f'{exp_config.results_dir}/fig_v_score_comparison.png', bbox_inches='tight')
plt.show()
print('✓ V-score comparison figure saved')

In [ ]:
"""
===============================================================================
Module: Tabular Data Synthesis & Publication Formatting
Reference: Experimental Results & Appendices
Description: 
    Automates the compilation and formatting of aggregated experimental 
    metrics into publication-ready tabular structures. This module translates 
    raw multi-seed validation data into standardized CSV formats for archival 
    and generates strictly compliant LaTeX table environments for direct 
    integration into IEEE academic manuscripts.
===============================================================================
"""

print('='*70)
print('  TABLE I — ABLATION STUDY RESULTS')
print('  (mirrors base KoMA Table III format, extended with V-score)')
print('='*70)

rows = []
for cfg_name, res in ab_results.items():
    rows.append({
        'Configuration':       CFG_LABELS[cfg_name],
        'Success @0':          f'{res.get("success_0ep", 0.0):.0%}',
        'Success @20':         f'{res["success_20ep"]:.0%}',
        'Success @40':         f'{res["success_40ep"]:.0%}',
        'Collision Rate':      f'{res["collision_rate"]:.1%}',
        'Safety /10':          f'{res["avg_safety"]:.2f}',
        'Efficiency /10':      f'{res["avg_efficiency"]:.2f}',
        'Mean V-Score':        f'{res["factual_consistency"]:.3f}',
    })

df_table1 = pd.DataFrame(rows).set_index('Configuration')
print(df_table1.to_string())
df_table1.to_csv(f'{exp_config.results_dir}/table1_ablation.csv')

# LaTeX
latex = df_table1.to_latex(caption='Ablation Study Results', label='tab:ablation')
with open(f'{exp_config.results_dir}/table1_ablation.tex', 'w') as f:
    f.write(latex)
print('\n✓ Table I saved (CSV + LaTeX)')


# ── TABLE II: Statistical summary ───────────────────────────
print('\n' + '='*70)
print('  TABLE II — STATISTICAL VALIDATION  (5 seeds)')
print('='*70)

stat_rows = []
for cfg in STAT_CONFIGS:
    arr  = np.array(stat_results[cfg])
    v_arr= np.array(stat_v[cfg])
    n    = len(arr)
    stat_rows.append({
        'Configuration':    CFG_LABELS[cfg],
        'Mean Success':     f'{arr.mean():.3f}',
        'Std':              f'{arr.std():.3f}',
        '95% CI':           f'[{arr.mean()-1.96*arr.std()/np.sqrt(n):.3f}, '
                            f'{arr.mean()+1.96*arr.std()/np.sqrt(n):.3f}]',
        'Mean V':           f'{v_arr.mean():.3f}',
        'V Std':            f'{v_arr.std():.3f}',
    })

df_table2 = pd.DataFrame(stat_rows).set_index('Configuration')
print(df_table2.to_string())
print(f'\n  Paired t-test (success): p={p_sr:.4f}  '
      f'{"✓ significant" if p_sr<0.05 else "✗ not significant"}')
print(f'  Paired t-test (V score): p={p_v:.4f}  '
      f'{"✓ significant" if p_v<0.05 else "✗ not significant"}')

df_table2.to_csv(f'{exp_config.results_dir}/table2_stats.csv')
print('\n✓ Table II saved')


# ── VALIDITY WARNING ─────────────────────────────────────────
if not llm.is_real_llm:
    print('\n' + '!'*65)
    print('  ⚠ ALL RESULTS ABOVE WERE GENERATED WITH MOCK LLM')
    print('  These numbers CANNOT be reported in an IEEE paper.')
    print('  Set your API key and re-run from Cell 14 onward.')
    print('!'*65)

In [ ]:
"""
===============================================================================
Module: Data Persistence & State Teardown
Reference: Reproducibility & Artifact Archival
Description: 
    Manages the final serialization and persistent storage of all generated 
    experimental artifacts, including statistical summaries, decision traces, 
    and publication-ready tables. Concurrently handles the safe flushing of 
    vector embedding caches and shared RAG memory states.
Execution:
    Ensures a clean environmental teardown, preventing memory leaks and 
    cross-contamination between experimental runs, while guaranteeing that 
    all empirical data required for peer-review reproducibility is securely 
    archived.
===============================================================================
"""

def _serialize(obj):
    if isinstance(obj, (np.floating, float)):  return float(obj)
    if isinstance(obj, (np.integer, int)):     return int(obj)
    if isinstance(obj, np.ndarray):            return obj.tolist()
    if isinstance(obj, Action):                return obj.name
    if isinstance(obj, dict):                  return {k: _serialize(v) for k, v in obj.items()}
    if isinstance(obj, list):                  return [_serialize(v) for v in obj]
    if isinstance(obj, RoundResult):
        d = asdict(obj)
        d['actions']  = [str(a) for a in obj.actions]
        return _serialize(d)
    return obj

save_data = {
    'metadata': {
        'timestamp':      datetime.now().isoformat(),
        'llm_provider':   api_config.provider.value,
        'is_real_llm':    llm.is_real_llm,
        'seeds':          exp_config.seeds,
        'train_rounds':   exp_config.train_rounds,
        'embed_cache_size': embedder.cache_size,
    },
    'ablation':     {k: _serialize({kk: vv for kk, vv in v.items()
                                    if kk != 'rounds' and kk != 'decision_trace'})
                     for k, v in ab_results.items()},
    'statistics': {
        'results': {k: _serialize(v) for k, v in stat_results.items()},
        'v_scores': {k: _serialize(v) for k, v in stat_v.items()},
        'p_success': float(p_sr),
        'p_vscore':  float(p_v),
    }
}

out_path = f'{exp_config.results_dir}/koma_rag_results.json'
with open(out_path, 'w') as f:
    json.dump(save_data, f, indent=2, default=_serialize)

embedder.flush()

print(f'✓ Results saved to {out_path}')
print(f'✓ Embedding cache: {embedder.cache_size} entries flushed to disk')
print(f'\n  LLM usage: {llm.call_count} calls, {llm.total_tokens} tokens')
print(f'  Provider:  {api_config.provider.value}  |  real={llm.is_real_llm}')

if not llm.is_real_llm:
    print('\n  *** REMINDER: re-run with a real API key for IEEE submission ***')